# Subsidence Calculator (Notebook Only)

This notebook provides a complete Jupyter-only workflow for:
- entering all required inputs,
- calculating subsidence outputs, and
- generating visualizations inline.

No Flask or web server is used in this notebook.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

from subsidence_calculator import create_report, format_report
from subsidence_visualization import plot_advanced_view

## Inputs

Enter panel points in the text box as one point per line using:

`easting,northing`

Example:

`0,0`
`100,0`
`100,100`
`0,100`

In [2]:
panel_points_text = widgets.Textarea(
    value="0,0\n100,0\n100,100\n0,100",
    description="Panel points:",
    layout=widgets.Layout(width="420px", height="140px"),
    style={"description_width": "110px"},
)

thickness = widgets.FloatText(value=2.5, description="h (m):", style={"description_width": "110px"})
depth_of_cover = widgets.FloatText(value=300.0, description="H (m):", style={"description_width": "110px"})
extraction_ratio = widgets.Dropdown(
    options=[("Longwall (1.0)", 1.0), ("Board and Pillar (0.7)", 0.7), ("Custom 0.9", 0.9)],
    value=1.0,
    description="Extraction:",
    style={"description_width": "110px"},
)
subsidence_factor = widgets.FloatText(value=0.5, description="a:", style={"description_width": "110px"})
new_ratio = widgets.FloatText(value=1.4, description="W(new)/H:", style={"description_width": "110px"})
angle_of_draw = widgets.FloatText(value=35.0, description="alpha (deg):", style={"description_width": "110px"})
mesh_spacing = widgets.FloatText(value=25.0, description="Mesh (m):", style={"description_width": "110px"})

show_matplotlib = widgets.Checkbox(value=True, description="Also show matplotlib multi-panel plots")
calculate_btn = widgets.Button(description="Calculate", button_style="primary")

summary_out = widgets.Output()
plot_out = widgets.Output()

In [3]:
def parse_panel_points(raw_text):
    points = []
    for line in raw_text.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = [p.strip() for p in line.split(',')]
        if len(parts) != 2:
            raise ValueError(f"Invalid point format: '{line}'. Use easting,northing")
        easting = float(parts[0])
        northing = float(parts[1])
        points.append((easting, northing))

    if len(points) < 3:
        raise ValueError("At least 3 panel points are required.")

    if len(set(points)) < 3:
        raise ValueError("At least 3 unique panel points are required.")

    return points


def build_plotly_map(report):
    points = report["points"]
    if not points:
        return None

    eastings = [p["easting"] for p in points]
    northings = [p["northing"] for p in points]
    subsidences_mm = [p["subsidence"] * 1000.0 for p in points]

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=eastings,
            y=northings,
            mode="markers",
            name="Grid points",
            marker={
                "size": 8,
                "color": subsidences_mm,
                "colorscale": "RdYlGn_r",
                "colorbar": {"title": "Subsidence (mm)"},
                "line": {"color": "black", "width": 0.4},
            },
            hovertemplate="E: %{x:.2f}<br>N: %{y:.2f}<br>S: %{marker.color:.2f} mm<extra></extra>",
        )
    )

    panel = report["inputs"]["panel_points"]
    if panel:
        px = [p["easting"] for p in panel] + [panel[0]["easting"]]
        py = [p["northing"] for p in panel] + [panel[0]["northing"]]
        fig.add_trace(
            go.Scatter(
                x=px,
                y=py,
                mode="lines+markers",
                name="Panel boundary",
                line={"color": "navy", "width": 3},
            )
        )

    fig.add_trace(
        go.Scatter(
            x=[report["inputs"]["center_easting"]],
            y=[report["inputs"]["center_northing"]],
            mode="markers",
            name="Centroid",
            marker={"size": 13, "color": "red", "symbol": "star"},
            hovertemplate="Centroid<extra></extra>",
        )
    )

    fig.update_layout(
        title="Notebook Subsidence Map",
        xaxis_title="Easting (m)",
        yaxis_title="Northing (m)",
        yaxis={"scaleanchor": "x", "scaleratio": 1},
        template="plotly_white",
        height=650,
    )

    return fig


def on_calculate(_):
    with summary_out:
        clear_output(wait=True)
    with plot_out:
        clear_output(wait=True)

    try:
        points = parse_panel_points(panel_points_text.value)

        report = create_report(
            points,
            float(thickness.value),
            float(depth_of_cover.value),
            float(extraction_ratio.value),
            float(subsidence_factor.value),
            float(new_ratio.value),
            float(angle_of_draw.value),
            float(mesh_spacing.value),
        )

        with summary_out:
            print(format_report(report))
            points_df = pd.DataFrame(report["points"])
            if points_df.empty:
                print("No influenced points were generated.")
            else:
                print("\nFirst 20 influenced points:")
                display(points_df.head(20))

        with plot_out:
            fig = build_plotly_map(report)
            if fig is None:
                print("No plot: zero influenced points for current parameters.")
            else:
                fig.show()

            if show_matplotlib.value:
                plot_advanced_view(report)

    except Exception as exc:
        with summary_out:
            print(f"Error: {exc}")


calculate_btn.on_click(on_calculate)

In [4]:
left = widgets.VBox(
    [
        panel_points_text,
        thickness,
        depth_of_cover,
        extraction_ratio,
        subsidence_factor,
        new_ratio,
        angle_of_draw,
        mesh_spacing,
        show_matplotlib,
        calculate_btn,
    ]
)

right = widgets.VBox([summary_out, plot_out])
ui = widgets.HBox([left, right])

display(ui)
on_calculate(None)

## Optional: Script-Style Run (No Widgets)

If widget rendering is unavailable in your environment, run this cell and adjust values directly in code.

In [6]:
panel_points = [(0.0, 0.0), (100.0, 0.0), (100.0, 100.0), (0.0, 100.0)]

report = create_report(
    panel_points=panel_points,
    thickness=2.5,
    depth_of_cover=300.0,
    extraction_ratio=1.0,
    subsidence_factor=0.5,
    new_ratio=1.4,
    angle_of_draw_deg=35.0,
    mesh_spacing=25.0,
)

print(format_report(report))
fig = build_plotly_map(report)
if fig is not None:
    fig.show()

SINGLE SEAM SUBSIDENCE REPORT
Panel boundary points (E, N):
  1. (0.0, 0.0)
  2. (100.0, 0.0)
  3. (100.0, 100.0)
  4. (0.0, 100.0)
Panel centroid: (50.0, 50.0)
Thickness h: 2.5
Depth of cover H: 300.0
Extraction ratio e: 1.0
Subsidence factor a: 0.5
NEW ratio W(new)/H: 1.4
Angle of draw alpha: 35.0 deg
Grid spacing: 25.0

s_max = h * e * a = 1.25 m (1250.0 mm)
Influence distance = H * tan(alpha) = 210.06 m
Nominal panel width from ratio = 420.0 m

Panel / influence limits:
  Panel west/east: 0.0 / 100.0
  Panel south/north: 0.0 / 100.0
  Grid west/east: -210.06226146291291 / 310.06226146291294
  Grid south/north: -210.06226146291291 / 310.06226146291294

Influenced points: 360
Max point subsidence: 1.25 m
Average point subsidence: 0.7511 m


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed